### Try-It Activity 18.1: Comparing Methods


This Try-It activity focuses on weighing the positives and negatives of different estimators and vectorization strategies for a text classification problem.  In order to consider each of these components, you should make use of the `Pipeline` and `GridSearchCV` objects in Scikit-Learn to try different combinations of vectorizers with different estimators.  For each of these, you also want to use the `.cv_results_` to examine the time for the estimator to fit the data.

### The Data

The dataset below is from [kaggle]() and contains a dataset named the "ColBert Dataset" created for this [paper](https://arxiv.org/pdf/2004.12765.pdf).  You are to use the text column to classify whether or not the text was humorous.  It is loaded and displayed below.


In [2]:
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import MultinomialNB

In [3]:
nltk.download("wordnet")
nltk.download('omw-1.4')
nltk.download("punkt_tab")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
df = pd.read_csv('/content/sample_data/dataset-minimal.csv')

In [5]:
print(df.columns )
df["humor"].value_counts()

Index(['text', 'humor'], dtype='object')


,count
humor,
False,50021
True,49978


In [6]:
print(df[df["humor"]==True].head(5))
df[df["humor"]==False].head(5)

                                                 text  humor
2   What do you call a turtle without its shell? d...   True
6   What is a pokemon master's favorite kind of pa...   True
7   Why do native americans hate it when it rains ...   True
9       My family tree is a cactus, we're all pricks.   True
13  How are music and candy similar? we throw away...   True


,text,humor
0,"Joe biden rules out 2020 bid: 'guys, i'm not r...",False
1,Watch: darvish gave hitter whiplash with slow ...,False
3,5 reasons the 2016 election feels so personal,False
4,"Pasco police shot mexican migrant from behind,...",False
5,"Martha stewart tweets hideous food photo, twit...",False


#### Task


**Text preprocessing:** As a pre-processing step, perform both `stemming` and `lemmatizing` to normalize your text before classifying. For each technique use both the `CountVectorize`r and `TfidifVectorizer` and use options for stop words and max features to prepare the text data for your estimator.

**Classification:** Once you have prepared the text data with stemming lemmatizing techniques, consider `LogisticRegression`, `DecisionTreeClassifier`, and `MultinomialNB` as classification algorithms for the data. Compare their performance in terms of accuracy and speed.

Share the results of your best classifier in the form of a table with the best version of each estimator, a dictionary of the best parameters and the best score.

In [7]:
stemmer = PorterStemmer()
def stemmer_func(text):
    '''
    This function takes in a string of text and returns
    a string of stemmed text.

    Arguments
    ---------
    text: str
        string of text to be stemmed

    Returns
    -------
    str
       string of stemmed words from the text input
    '''
    tokenized_text=nltk.word_tokenize(text)
    # Corrected: ensure it returns a single string, not a list
    stemmed_text=(' ').join(stemmer.stem(w) for w in tokenized_text)
    return  stemmed_text

### ANSWER CHECK
text = 'The computer did not compute the answers correctly.'
print(stemmer_func(text)) #should return --> the comput did not comput the answer correctli .

the comput did not comput the answer correctli .


In [66]:
# Reload the original dataframe to ensure 'text' column contains strings
df = pd.read_csv('/content/sample_data/dataset-minimal.csv')

# Apply the corrected stemmer_func to create a new stemmed text series
stemmed_X = df['text'].apply(stemmer_func)
print('First 2 stemmed texts (should be strings now):')
print(stemmed_X.head(2))

First 2 stemmed texts (should be strings now):
0    joe biden rule out 2020 bid : 'guy , i 'm not ...
1    watch : darvish gave hitter whiplash with slow...
Name: text, dtype: object


In [67]:
# Use the newly created stemmed_X for feature engineering
X = stemmed_X
y = df["humor"]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
print('X_train and X_test created from stemmed data.')

X_train and X_test created from stemmed data.


In [10]:
#Then use CountVectorizer and TfidifVectorizer each plus stop words option and max_features for preparing data for the estimator.
pipeline = Pipeline([
    ("count_vec", CountVectorizer()),
     ("lr", LogisticRegression())
])
pipeline.fit(X_train, y_train)
stem_score_count_vec=pipeline.score(X_test, y_test)
print(f'Accuracy with CountVectorizer and Logistic Regression on stemmed data: {stem_score_count_vec}')

Accuracy with CountVectorizer and Logistic Regression on stemmed data: 0.92048


In [69]:
# Use word lemmatizer
lemma = WordNetLemmatizer()
def lemmatizer_func(text):
    '''
    This function takes in a string of text and returns
    a string of lemmatized text.

    Arguments
    ---------
    text: str
        string of text to be lemmatized

    Returns
    -------
    str
       string of lemmatized words from the text input
    '''
    tokenized_text = nltk.word_tokenize(text)
    # Corrected: ensure it returns a single string, not a list
    lemmatized_text = ' '.join(lemma.lemmatize(w) for w in tokenized_text)
    return lemmatized_text


### ANSWER CHECK
text = 'All the calculators computes these answers quite fast.'
print(lemmatizer_func(text)) #should return --> All the calculator computes these answer quite fast .

All the calculator computes these answer quite fast .


### Try Lemmatization with the same approach
Apply the corrected lemmatizer func to create a new lemmatized text series

In [68]:
lemma_X = df['text'].apply(lemmatizer_func)
print('First 2 lemmatized texts (should be strings now):')
print(lemma_X.head(2))

First 2 lemmatized texts (should be strings now):
0    Joe biden rule out 2020 bid : 'guys , i 'm not...
1    Watch : darvish gave hitter whiplash with slow...
Name: text, dtype: object


In [70]:
X_lemma=lemma_X
X_train_lemma, X_test_lemma, y_train_lemma, y_test_lemma = train_test_split(X_lemma, y, random_state=42)
print('X_train and X_test created from lemmatized data.')

X_train and X_test created from lemmatized data.


In [71]:
lemma_pipe = Pipeline([
    ("count_vec_l",CountVectorizer()),
    ("lr",LogisticRegression())
])
lemma_pipe.fit(X_train_lemma, y_train_lemma)
lemma_score_count_vec=lemma_pipe.score(X_test_lemma, y_test_lemma)
print(f'Accuracy with CountVectorizer and Logistic Regression on lemmatized data: {lemma_score_count_vec}')

Accuracy with CountVectorizer and Logistic Regression on lemmatized data: 0.92348


In [39]:
result_all_models= pd.DataFrame({'model': ['Logistic', 'Decision Tree', 'Bayes'],
             'best_params': ['', '', ''],
             'best_score': ['', '', '']}).set_index('model')
result_all_models

,best_params,best_score
model,,
Logistic,,
Decision Tree,,
Bayes,,


### Move towards Grid search and testing various models.

**Text preprocessing:** As a pre-processing step, perform both `stemming` and `lemmatizing` to normalize your text before classifying. For each technique use both the `CountVectorize`r and `TfidifVectorizer` and use options for stop words and max features to prepare the text data for your estimator.

**Classification:** Once you have prepared the text data with stemming lemmatizing techniques, consider `LogisticRegression`, `DecisionTreeClassifier`, and `MultinomialNB` as classification algorithms for the data. Compare their performance in terms of accuracy and speed.

Share the results of your best classifier in the form of a table with the best version of each estimator, a dictionary of the best parameters and the best score.

In [61]:
# Define the parameter grid for combinations of models - Logistic Regression , Naive Bayes Multionomial classifier
# Define models -
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
     "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

#Define  param grid for count vectorizer
cvect_params = {'cvect__max_features': [100, 500, 1000, 2000],
         'cvect__stop_words': ['english', None]}

#Define  param grid for tfidf vectorizer
tfidf_params = {'tfidf__max_features': [100, 500, 1000, 2000],
         'tfidf__stop_words': ['english', None]}

In [72]:
#Reinstate the training and test data for stemming purposes as they have been converted to lemmatized ones
X_stem = stemmed_X
X_train_stem, X_test_stem, y_train_stem, y_test_stem = train_test_split(X, y, random_state=42)
print('X_train_stem_ and X_test_stem created from stemmed data.')
y_test_stem.head(2)

X_train_stem_ and X_test_stem created from stemmed data.


,humor
26002,True
80420,False


In [63]:
import timeit
from timeit import default_timer as timer
from datetime import datetime
pipeline, lemma_pipe

(Pipeline(steps=[('count_vec', CountVectorizer()), ('lr', LogisticRegression())]),
 Pipeline(steps=[('count_vec_l', CountVectorizer()),
                 ('lr', LogisticRegression())]))

In [64]:
for name , model in models.items():
  print(f"Model = {model}")

Model = LogisticRegression(max_iter=2000, random_state=42)
Model = MultinomialNB()
Model = DecisionTreeClassifier(random_state=42)


In [78]:
# compare X_train and X_train_stem if they are equal or not
print(X_train_stem.equals(X_train_lemma))
print(X_test_stem.equals(X_test_lemma))

False
False


In [ ]:
### Stemming based grid search on param grid for CountVectorizer
best_models = {}
results_cv = []

for name, model in models.items():
  # #When you call grid.score(X_test, y_test) after fitting a GridSearchCV object,
  # it uses the best estimator found during the grid search to evaluate the performance on the test set.
  # The GridSearchCV object automatically refits the best estimator on the entire training dataset
  #(unless refit=False is specified during initialization),
  # and the score method then uses this refitted bestestimator.
    pipe = Pipeline([("cvect", CountVectorizer()),("model",model)])
    grid = GridSearchCV(pipe, param_grid=cvect_params)
    start_time = timer()
    print(f"Tuning {name}...at ", datetime.now(),  " UTC")
    grid.fit(X_train_stem, y_train_stem)
    train_time_model = timer() - start_time
    best_models[name] = grid.best_estimator_
    grid.score(X_test, y_test)

    # Store results
    results_cv.append({"model": f"Tuned {name}", "best_f1_cv": grid.best_score_, "best_params": grid.best_params_, "Train Time": train_time_model})



Tuning Logistic Regression...at  2026-04-26 17:59:58.432343  UTC
Tuning Naive Bayes...at  2026-04-26 18:00:58.378589  UTC


In [49]:
results_cv_df = pd.DataFrame(results_cv).sort_values("best_f1_cv", ascending=False).reset_index(drop=True)
results_cv_df

KeyError: 'best_f1_cv'

In [ ]:
### Lemmatization based grid search on param grid for CountVectorizer

In [ ]:
### Stemming based grid search on param grid for CountVectorizer

In [ ]:
### Lemmatization based grid search on param grid for TFIDFVectorizer